In [1]:
from workloads.common import *
from roofline.roofline_model import *

In [5]:
test_layer = {"nix": 256, "niy": 256, "nif": 3, "nof": 64, "kernel": 7, "stride":2,  "type": "pw_conv"}

In [6]:
# config
save_path = 'files/test_layer'

board_part = 'zcu102'
double_buff=False
num_hp = 1
t_factor = 2
bits = 9
buswidth = 1024

# search design for a layer
pair_ls, comp_bnd, bw_bnd, solution = DSE_layer(\
    test_layer, board_part, 'test_layer', bits, buswidth, t_factor, save_path, \
    double_buff=double_buff, num_hp = num_hp)

100%|██████████| 1048576/1048576 [00:05<00:00, 183705.20it/s]


In [7]:
solution

((223.10515163094453, 68.31719534626039), (64, 64, 4, 64), (620, 256))

In [5]:
from math import ceil
from workloads import vgg16, common
from roofline.roofline_model import *
def spatial_access_size(R, C, M, N, K, S, Tr, Tc, Tm, Tn):
    Tr = Tr if R > Tr else R
    Tc = Tc if C > Tc else C
    
    beta_in = Tn * (S*Tr + K-S) * (S*Tc + K-S)
    beta_wgt = Tn * Tm * K * K
    beta_out = Tm * Tr * Tc
    
    alpha_in = ceil(M/Tm) * ceil(N/Tn) * ceil(R/Tr) * ceil(C/Tc)
    alpha_wgt = ceil(M/Tm) * ceil(N/Tn)
    alpha_out = 2 * ceil(R/Tr) * ceil(C/Tc) * ceil(M/Tm) * ceil(N/Tn)
    
    num_ext_access = alpha_in * beta_in + alpha_wgt * beta_wgt + alpha_out * beta_out
    
    return (num_ext_access*4.)/(10**9)

def channel_access_size(R, C, M, N, K, S, Tr, Tc, Tm, Tn):
    Tr = Tr if R > Tr else R
    Tc = Tc if C > Tc else C
    
    beta_in = Tn * (S*Tr + K-S) * (S*Tc + K-S)
    beta_wgt = Tn * Tm * K * K
    beta_out = Tm * Tr * Tc
    
    alpha_in = ceil(M/Tm) * ceil(N/Tn) * ceil(R/Tr) * ceil(C/Tc)
    alpha_wgt = ceil(M/Tm) * ceil(N/Tn) * ceil(R/Tr) * ceil(C/Tc)
    alpha_out = ceil(M/Tm) * ceil(R/Tr) * ceil(C/Tc)
    
    num_ext_access = alpha_in * beta_in + alpha_wgt * beta_wgt + alpha_out * beta_out
    
    return (num_ext_access*4.)/(10**9)

/Volumes/Transcend/git/DesignSpaceExplore/roofline/roofline_model.py:26: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if bits is 8:
/Volumes/Transcend/git/DesignSpaceExplore/roofline/roofline_model.py:28: SyntaxWarning: "is" with a literal. Did you mean "=="?
  elif bits is 16:
/Volumes/Transcend/git/DesignSpaceExplore/roofline/roofline_model.py:30: SyntaxWarning: "is" with a literal. Did you mean "=="?
  elif bits is 32:
/Volumes/Transcend/git/DesignSpaceExplore/roofline/roofline_model.py:39: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if bits is 8:
/Volumes/Transcend/git/DesignSpaceExplore/roofline/roofline_model.py:41: SyntaxWarning: "is" with a literal. Did you mean "=="?
  elif bits is 16:
/Volumes/Transcend/git/DesignSpaceExplore/roofline/roofline_model.py:43: SyntaxWarning: "is" with a literal. Did you mean "=="?
  elif bits is 32:
/Volumes/Transcend/git/DesignSpaceExplore/roofline/roofline_model.py:162: SyntaxWarning: "is" with a literal. Did you mean "

In [2]:
common.resolutions['original']

net = vgg16.network(common.resolutions['original'], 101)
tiling_factor = [4,8,16,32,64]

spatial_size_ls = []
channel_size_ls = []

for t in tiling_factor:
    spatial = 0
    channel = 0
    for l in net.values():
        R = l['nix']; C = l['niy']; M = l['nif']; N = l['nof']; K=l['kernel']; S = 1; 
        Tr = t; Tc = t; Tm = 16; Tn = 32;
        
        spatial += spatial_access_size(R, C, M, N, K, S, Tr, Tc, Tm, Tn)
        channel += channel_access_size(R, C, M, N, K, S, Tr, Tc, Tm, Tn)
        
    spatial_size_ls.append(spatial)
    channel_size_ls.append(channel)

In [3]:
spatial_size_ls

[1.912754176, 1.520586752, 1.3512417279999995, 1.280135168, 1.2570828799999998]

In [4]:
channel_size_ls

[6.468141056000001, 2.2256025600000005, 1.093664768, 0.80314368, 0.74293248]

In [7]:
beta_in = Tn * (S*Tr + K-S) * (S*Tc + K-S)
beta_wgt = Tn * Tm * K * K
beta_out = Tm * Tr * Tc
bram_usage(beta_in, beta_wgt, beta_out, Tn, Tm, bits = 32, double_buff=False)

1312